# GLiNER2 도메인 온톨로지 SFT: 군부대 보안 감사 (1단계 시험)

[한국어 미세조정](../notebooks/02_korean_fine_tuning.ipynb)과 [LLM 증류](03_llm_distillation.ipynb)는 사람·기관·장소 3타입의 엔티티만 다뤘다. 본 실험은 사용자가 정의한 **도메인 온톨로지 전체**(엔티티 11타입, 관계 10종, 구조화 레코드 3종)를 정답 데이터로 지도 미세조정(SFT)했을 때 정답지 대비 얼마나 정확히 추출하는지 측정한다.

시나리오는 **가상의** 군부대 보안 감사이다. 감사관이 부대의 문서와 기록(점검 보고서, 사고 경위서, 대장, 회의록)을 분석해 지식 그래프로 만드는 상황을 가정한다. 보안 영역은 인원·문서·시설·정보통신 네 가지이며, 1단계는 네 영역을 하나의 **통합 어댑터**로 학습한다. 군종·수검 기관별 특화 어댑터는 2단계에서 다룬다.

> 이 노트북은 30개 시나리오(문서 140개)로 파이프라인 전체를 검증하는 **시험 규모** 결과이다. 모든 데이터는 가상이며 실존 부대·인원·비밀 내용과 무관하다.

## 결과 요약

> 아래 수치는 2026-11-13 실행 결과(`security_audit_sft_pilot.json`)이다. 학습 20개 시나리오(문서 90개), 오류 분석용 검증 4개 시나리오(문서 18개), 평가 6개 시나리오(문서 32개). Darwin arm64 CPU.

| 평가 대상 | 지표 | 목표 | 학습 전 (zero-shot) | **통합 어댑터** | 달성 |
|---|---|---|---|---|---|
| 엔티티 | 범위+타입 정확 일치 F1 | 0.90 | 0.672 | **0.945** | ✓ (아래 주의) |
| └ 학습에 없던 표면형만 | 같은 지표 | – | 0.566 | **0.905** | – |
| 관계 | (head, 관계, tail) 정확 일치 F1 | 0.80 | 0.333 | **0.656** | ✗ |
| 레코드 필드 | 필드 값 정확 일치 F1 | 0.80 | 0.601 | **0.842** | ✓ |
| 사건 유형 | 정확도 (평가 레코드 6개) | 0.90 | 0.833 | **0.667** | ✗ (판단 불가) |

- **엔티티 점수는 학습 데이터와 겹치는 이름 덕분에 높게 나왔다.** 가상 엔티티 사전이 작아서, 평가 데이터의 사람 외 엔티티 표면형 183개 중 166개(91%)가 학습 데이터에도 그대로 나온다. 학습에 없던 표면형(대부분 사람 이름)만 따로 재면 0.566 → 0.905로, 여전히 크게 오르지만 전체 점수(0.945)보다는 낮다.
- **레코드 필드는 목표를 넘었다**(0.601 → 0.842). 다만 위반 규정처럼 값이 정해진 목록에서만 나오는 필드는 같은 이유로 쉬워진다.
- **관계는 0.333 → 0.656으로 올랐지만 목표에 못 미쳤다.** 오답의 절반가량은 이미 정답으로 연결된 사람·대상 쌍에 다른 관계(예: 열람한 사람에게 반출·관리 책임까지)를 덧붙이는 오류이다.
- **사건 유형 정확도는 판단할 수 없다.** 평가 레코드가 6개뿐이고, 선택형 필드는 정답 단어가 본문에 적혀 있는지에 크게 좌우된다(아래 레코드 절).
- 시험 데이터의 한계: 문서 종류가 모두 정형화된 감사 보고 문서이고, `저장매체 반출입 대장`조차 표 형식이 아닌 서술형으로 쓰인 경우가 많다. 이 결과는 "같은 사전과 비슷한 서식의 문서"에서의 성능이다.

## 준비

### 경고 출력 설정

라이브러리가 출력하는 안내성 경고를 숨긴다.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch.distributed.elastic").setLevel(logging.ERROR)

### 프로젝트 경로와 결과 파일

프로젝트 루트를 찾아 모듈 검색 경로에 추가한다. 실험은 [`scripts/run_security_audit_sft.py`](../scripts/run_security_audit_sft.py)로 실행하며(CPU 약 15분), 이 노트북은 저장된 결과를 불러와 분석한다.

In [ ]:
import json
from pathlib import Path
import sys

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
RESULT_PATH = PROJECT / "experiments/results/security_audit_sft_pilot.json"
DATA = PROJECT / "experiments/data/security_audit"
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    print("프로젝트 루트에서 실행한다: uv run python scripts/run_security_audit_sft.py")

result = json.loads(RESULT_PATH.read_text())
print(f"✓ Result file: {RESULT_PATH.relative_to(PROJECT)}")
print(f"  {result['run']['date']}  docs={result['run']['docs']}  scenarios={result['run']['scenarios']}")

## 온톨로지

온톨로지는 [create-context-graph](https://create-context-graph.dev/docs/reference/ontology-yaml-schema)의 YAML 형식(`entity_types`의 `pole_type`·`subtype`, `relationships`의 `source`·`target`)을 따르고, 실험용 키(`description_ko`, `extract`, `records`)를 더했다. 파일은 [`config/domains/military-security-audit.yaml`](../config/domains/military-security-audit.yaml), 로더는 [`config/security_audit.py`](../config/security_audit.py)이다. GLiNER의 레이블, JointIE 관계 스키마, 구조화 레코드 스키마를 모두 이 파일에서 만든다.

In [ ]:
from config.security_audit import load_ontology

ontology = load_ontology()
print("entities (extract):")
for t in ontology.entity_types.values():
    print(f"  {'✓' if t.extract else '·'} {t.label:<19} {t.pole_type:<13} {t.description_ko[:42]}")
print("relations (extract):")
for r in ontology.relations:
    print(f"  {'✓' if r.extract else '·'} {r.type:<16} {'|'.join(r.sources)} → {'|'.join(r.targets)}")
print("records:")
for name, record in ontology.records.items():
    print(f"  {name}: " + ", ".join(f.name + ('*' if f.choices else '') for f in record.fields))

- 엔티티는 POLE+O 기본 범주 위의 14타입이다. 그중 이름이 있는 **11타입을 GLiNER가 추출**하고, 사건(SecurityIncident)·발견 사항(Finding)·조치(CorrectiveAction)는 문서 단위로 만든다(경위서 1건 = 사고 1건, 점검표 항목 1개 = 발견 사항 1개, 조치표 1행 = 조치 1개).
- 관계 17종 중 엔티티 사이의 **10종을 관계 추출**로, 사건 노드와 이어지는 7종은 레코드 필드로 표현한다.
- 레코드는 incident·finding·action 3종이며, `*`는 선택형 필드(사건 유형, 보안 영역, 조치 유형)이다.
- 표기 규칙: Person은 문서에 적힌 계급을 포함한다(`김민수 상병`, `상병 김민수`, `김 상병`). 직책은 별도 타입 Position(`보안담당관`)으로 두고 HOLDS_POSITION으로 사람과 잇는다.

## 데이터 생성

정답이 일관되도록 **시나리오를 먼저 만들고 문서를 나중에 쓴다**.

```text
엔티티 사전 (가상의 이름·계급·부대·시설·매체·문서·체계·규정)
  ↓ make_scenarios.py (seed 42)
시나리오 그래프 30개 (사건 유형 6종 × 5, 보안 영역 4종)          ← 정답 그래프
  ↓ 서브에이전트 5개가 writing_prompt.md 지시로 작성
문서 140개 ([[타입|표면형]] 인라인 표시 + 관계 + 레코드 정답)   ← 학습·평가 데이터
  ↓ validate_docs.py: 표면형-타입 일관성, 관계 타입 규칙, 레코드 값이 원문에 있는지 (오류 0)
```

- 문서 종류는 8가지이다: 보안점검 결과보고서, 보안사고 경위서, 출입기록 점검 결과, 저장매체 반출입 대장, 비밀문서 관리대장 점검표, 보안교육 결과, 정보체계 접근권한 점검 결과, 보안대책 회의록.
- **평가 데이터는 학습과 떼어 놓았다.** 평가 시나리오 6개는 학습에 쓰지 않았고, `보안대책 회의록`은 평가에만 있으며, 평가 문서는 더 간결한 문체("~함/~임", `일병 윤하율` 같은 어순, `윤 일병` 같은 줄임)로 별도 작성했다. 다만 엔티티 사전은 학습과 평가가 공유한다(아래 결과 해석 참고).
- 학습 시나리오 중 4개를 오류 분석용 검증 데이터(dev)로 떼어 두었다.
- 표시를 지운 원문은 [`experiments/data/security_audit/plain/`](data/security_audit/plain/)에서 볼 수 있다.

아래는 평가 문서 하나의 원문과 정답이다.

In [ ]:
from config.annotation import parse_marked

doc = json.loads((DATA / "docs/batch_5.jsonl").read_text().splitlines()[0])
print(f"[{doc['doc_type']}] {doc['doc_id']}\n")
for i, marked in enumerate(doc["paragraphs"]):
    print(f"p{i} marked: {marked}")
print("\nrelations:")
for r in doc["relations"]:
    print(f"  p{r['paragraph']}: {r['head']} -{r['type']}→ {r['tail']}")
print("\nrecords:")
for r in doc["records"]:
    print(f"  {r['record']}: {r['fields']}")

## 데이터 통계

In [ ]:
stats = result["data_stats"]
print(f"training examples: paragraphs {stats['paragraph_examples']}, records {stats['record_examples']}")
print(f"dropped from training: misaligned entity spans {stats['misaligned']}, "
      f"relations {stats['relations_dropped']}, record values {stats['fields_dropped']}")
print(f"training: {result['training']['total_time_seconds']:.0f}s, {result['training']['total_steps']} steps")

| 항목 | 학습 | 검증(dev) | 평가 |
|---|---|---|---|
| 시나리오 / 문서 | 20 / 90 | 4 / 18 | 6 / 32 |
| 엔티티 정답 | 760 | 171 | 272 |
| 관계 정답 | 291 | 67 | 97 |
| 레코드 (incident / finding / action) | 20 / 39 / 38 | 4 / 9 / 7 | 6 / 17 / 12 |

- 학습 예시는 문단 470개(엔티티·관계)와 레코드 97개이다. 학습 예시에는 추론 때와 같은 한국어 레이블 설명을 넣었다. 2 epoch, 140 단계이며, 이번 실행의 학습 시간은 CPU에서 약 26분이었다(같은 설정의 이전 실행은 약 2분이라, 실행 당시 다른 작업 부하가 있었던 것으로 보인다).
- **관계 291개 중 54개(19%)가 학습에서 빠졌다.** GLiNER2는 관계의 머리·꼬리와 레코드 값을 위치가 아닌 문자열로 받는다. 이 값의 토큰을 문장에서 찾지 못하면 학습이 실패하므로, 라이브러리의 정렬 검사(`find_unalignable_entities`)로 미리 걸러 뺐다. 형태소 분할기가 이름을 문장 안에서와 따로 떼었을 때 다르게 나누는 경우로 보이지만, 어떤 이름에 몰려 있는지는 확인하지 않았다. 레코드 값 22개도 같은 이유로 빠졌다.
- 실제로 학습에 쓰인 관계 수는 HOLDS_POSITION 50, BELONGS_TO 36, ACCESSED 36, POSITION_OF 30, INSPECTED 28, RESPONSIBLE_FOR 21, HOLDS 11, CARRIED_OUT 9, STORED_ON 9, LOCATED_AT 7이다.
- 엔티티는 문자 위치를 함께 주므로 이 문제가 없다. 형태소 경계와 맞지 않아 빠진 엔티티는 5개뿐이다.
- Organization은 시나리오 생성기가 만들지 않아 정답이 없다. 아래 표에서 Organization은 측정값이 아니다.

## 조건별 성능

학습 전(zero-shot, 레이블 설명만 제공)과 통합 어댑터를 같은 평가 데이터로 비교한다. 오류 분석과 판단은 dev로 하고, 평가 데이터는 최종 점수 보고에만 썼다.

In [ ]:
for name in ("zero_shot", "adapter"):
    for split in ("dev", "eval"):
        x = result["conditions"][name][split]
        print(f"{name:>9}/{split:<4}  entities {x['entities']['overall']['f1']:.3f}  "
              f"relations {x['relations']['overall']['f1']:.3f}  "
              f"record fields {x['records']['overall']['f1']:.3f}  choice {x['records']['choice_overall']:.3f}")

| 조건 | 데이터 | 엔티티 F1 | 관계 F1 | 레코드 필드 F1 | 사건 유형 정확도 |
|---|---|---|---|---|---|
| zero-shot | dev | 0.600 | 0.179 | 0.655 | 0.75 (4건) |
| zero-shot | 평가 | 0.672 | 0.333 | 0.601 | 0.83 (6건) |
| **통합 어댑터** | dev | 0.936 | 0.671 | 0.902 | 0.50 (4건) |
| **통합 어댑터** | 평가 | **0.945** | **0.656** | **0.842** | **0.67 (6건)** |

dev와 평가의 점수가 비슷하다. 그러나 dev도 평가도 학습과 같은 엔티티 사전에서 나왔으므로, 이것만으로는 문체나 서식에 대한 일반화를 판단할 수 없다.

### 엔티티 타입별 (평가)

In [ ]:
for name in ("zero_shot", "adapter"):
    entities = result["conditions"][name]["eval"]["entities"]
    print(f"{name:>9}: " + "  ".join(f"{label} {entities[label]['f1']:.2f}"
                                   for label in ontology.extract_labels() if label != "Organization"))

| 타입 | zero-shot | 어댑터 | 평가 정답 중 학습에 없던 표면형 |
|---|---|---|---|
| Person | 0.82 | 0.96 | 87 / 89 |
| Position | 0.79 | 0.94 | 0 / 31 |
| Unit | 0.58 | 0.99 | 7 / 42 |
| Facility | 0.68 | 0.92 | 0 / 35 |
| Credential | 0.37 | 1.00 | 3 / 6 |
| ClassifiedDocument | 0.26 | 0.78 | 0 / 14 |
| StorageMedium | 0.12 | 1.00 | 7 / 7 |
| InformationSystem | 0.47 | 1.00 | 0 / 7 |
| Regulation | 0.57 | 0.88 | 0 / 27 |
| Inspection | 0.97 | 1.00 | 0 / 14 |

- 학습 전에도 사람(0.82)과 행사명(0.97)은 잘 잡지만, 도메인 고유 타입(저장매체 0.12, 비밀 문서 0.26, 인가·계정 0.37)은 거의 못 잡는다.
- **학습 후 높은 점수의 상당 부분은 같은 이름을 외운 효과일 수 있다.** 직책·시설·규정·비밀 문서·정보체계·행사명은 평가 정답이 모두 학습 데이터에 나온 표면형이다. 이 타입들의 점수는 "처음 보는 이름을 그 타입으로 알아보는가"가 아니라 "학습한 이름을 다시 찾는가"에 가깝다.
- 처음 보는 표면형이 있는 타입에서의 결과가 일반화의 근거이다. 사람(87개, 0.96)은 학습에 없던 이름도 잘 잡고, 저장매체(7개 모두 새 관리번호, 1.00)도 `업무용 USB-124`처럼 처음 보는 번호를 잡았다. 다만 저장매체는 이름 패턴(`보안USB-`, `외장하드-`, `업무용 USB-`)이 학습과 같다.

### 학습에서 본 표면형과 처음 보는 표면형 (평가)

In [ ]:
for name in ("zero_shot", "adapter"):
    by_seen = result["conditions"][name]["eval"]["entities_by_seen"]
    print(f"{name:>9}: seen ({by_seen['seen_gold']}) F1 {by_seen['seen']['overall']['f1']:.3f}   "
          f"unseen ({by_seen['unseen_gold']}) F1 {by_seen['unseen']['overall']['f1']:.3f}")

| 조건 | 학습에서 본 표면형 (168개) | 처음 보는 표면형 (104개) |
|---|---|---|
| zero-shot | 0.774 | 0.566 |
| 어댑터 | 0.972 | **0.905** |

처음 보는 표면형에서도 0.566 → 0.905로 크게 오른다. 평가 데이터의 104개는 대부분 사람 이름(87개)이라 "도메인 고유 타입의 처음 보는 이름"에 대한 근거는 부족하다. 전체 규모 실험에서는 학습과 평가의 엔티티 사전을 나눠야 한다.

### 관계 종류별 (평가)

In [ ]:
relations = result["conditions"]["adapter"]["eval"]["relations"]
for kind, s in sorted(relations.items(), key=lambda kv: -kv[1]["support"]):
    print(f"{kind:<16} F1 {s['f1']:.2f}  P {s['precision']:.2f}  R {s['recall']:.2f}  gold {s['support']}")

| 관계 | 학습에 쓰인 수 | 평가 정답 | 어댑터 F1 |
|---|---|---|---|
| HOLDS_POSITION | 50 | 28 | 0.81 |
| ACCESSED | 36 | 21 | 0.65 |
| RESPONSIBLE_FOR | 21 | 14 | 0.46 |
| INSPECTED | 28 | 9 | 1.00 |
| BELONGS_TO | 36 | 6 | 0.50 |
| CARRIED_OUT | 9 | 5 | 0.33 |
| POSITION_OF | 30 | 4 | 0.73 |
| HOLDS | 11 | 4 | 1.00 |
| STORED_ON | 9 | 4 | 0.67 |
| LOCATED_AT | 7 | 2 | 0.00 |

학습에 쓰인 수와 점수가 일관된 관계를 보이지는 않는다. BELONGS_TO는 36개로 학습했지만 0.50이고, STORED_ON은 9개로 0.67이다. 평가 정답이 관계마다 2~28개라 점수 자체의 편차도 크다.

**오답 분석 (dev).** 오류 분석용 검증 데이터에서 어댑터가 낸 관계 중 정답에 없는 38개를 분류했다.

| 분류 | 개수 | 예 |
|---|---|---|
| 정답에 있는 쌍에 다른 관계를 덧붙임 | 18 | 정답 `임지우 일병 -ACCESSED→ Ⅱ급 비밀 경계작전 명령`에 `-CARRIED_OUT→`, `-RESPONSIBLE_FOR→`까지 냄 |
| 머리글 문단 | 5 | `점검 대상: 제6008공병단 정보과`에서 `-INSPECTED→` (대부분 정답 누락) |
| 정답 관계가 없는 문단 | 6 | 정답 누락과 오답이 섞임 |
| 엔티티 범위에 조사가 붙음 | 2 | `안지원 상사이 -BELONGS_TO→ …` |
| 같은 사람의 다른 표기 | 1 | 정답 `안 상사`, 예측 `안지원 상사` |
| 기타 | 6 | 관리 책임자에게 출입 관계, 사람을 잘못 고름 등 |

- **가장 큰 오류는 관계 종류를 구분하지 못하는 것이다.** 사람과 문서·매체·정보체계가 한 문장에 나오면 열람·반출·관리 책임을 함께 붙인다. GLiNER는 관계 종류마다 독립적으로 점수를 내므로 같은 쌍에 여러 관계가 동시에 나올 수 있다.
- **정답 누락은 일부이다.** 머리글의 점검 대상·소속처럼 문서에서 확인할 수 있는데 작성자가 정답에 넣지 않은 관계가 5~7개 있다. 관계 정밀도를 일부 낮췄지만 주된 원인은 아니다.

### 레코드 필드 (평가)

In [ ]:
for name in ("zero_shot", "adapter"):
    records = result["conditions"][name]["eval"]["records"]
    fields = "  ".join(f"{k} {v['f1']:.2f}" for k, v in records["fields"].items())
    choices = "  ".join(f"{k} {v:.2f}" for k, v in records["choice_accuracy"].items())
    print(f"{name:>9}: {fields}\n{'':>11}choice: {choices}")

| 레코드.필드 | 평가 정답 | zero-shot | 어댑터 |
|---|---|---|---|
| finding.대상 | 35 | 0.32 | 0.73 |
| finding.위반 규정 | 17 | 0.48 | 0.94 |
| action.담당 | 12 | 1.00 | 1.00 |
| action.기한 | 12 | 0.92 | 0.92 |
| incident.관련자 | 6 | 0.27 | 0.56 |
| incident.일자 | 6 | 1.00 | 1.00 |
| incident.장소 | 6 | 0.71 | 0.80 |
| incident.위반 규정 | 6 | 0.91 | 1.00 |
| **선택형: incident.유형** | 6 | 0.83 | 0.67 |
| **선택형: finding.영역** | 17 | 0.53 | 0.53 |
| **선택형: action.조치 유형** | 12 | 1.00 | 1.00 |

- 발견 사항의 대상(0.32 → 0.73)과 위반 규정(0.48 → 0.94)이 가장 크게 올랐다. 위반 규정은 정해진 6개 이름 중 하나라서 쉬운 필드이다.
- `incident.관련자`(0.56)는 같은 사람의 다른 표기(`일병 윤하율` / `윤 일병`)가 정답과 다르면 오답이 되는 채점 방식의 영향도 받는다.

**선택형 필드는 정답 단어가 본문에 적혀 있는지에 좌우된다.** 정답 선택지가 레코드 본문에 글자 그대로 나오는 비율은 다음과 같다.

In [ ]:
for split, rates in result["choice_literal_rate"].items():
    print(f"{split:>5}: " + "  ".join(f"{k} {v:.0%}" for k, v in rates.items()))

| 필드 | 학습 | 평가 |
|---|---|---|
| action.조치 유형 | 100% | 100% |
| finding.영역 | 90% | 35% |
| incident.유형 | 40% | 17% |

- 조치 유형은 본문에 항상 적혀 있어 학습 전에도 1.00이다. 측정할 의미가 거의 없다.
- 보안 영역은 학습 문서의 90%에 `[정보통신보안]`처럼 적혀 있었지만, 평가 문서는 35%만 적혀 있어 학습 전후 모두 0.53에 머문다. 모델이 영역을 판단하기보다 적힌 단어를 고르는 데 의존한 것으로 보인다.
- 사건 유형은 평가 레코드가 6개뿐이라 판단하지 않는다. 전체 규모에서는 선택지 단어가 본문에 나오지 않는 레코드를 따로 집계해야 한다.

### 문서 종류와 보안 영역별 (평가, 어댑터)

In [ ]:
for key in ("by_doc_type", "by_area"):
    print(key)
    for value, s in result[key]["adapter"].items():
        rec = f"  record {s['record_fields_f1']:.2f}" if "record_fields_f1" in s else ""
        print(f"  {value:<16} entities {s['entities_f1']:.2f}  relations {s['relations_f1']:.2f}{rec}")

| 문서 종류 | 엔티티 F1 (zero-shot → 어댑터) | 관계 F1 | 레코드 필드 F1 |
|---|---|---|---|
| 보안교육 결과 | 0.64 → 1.00 | 0.67 → 0.80 | – |
| 저장매체 반출입 대장 | 0.63 → 0.97 | 0.67 → 0.73 | – |
| **보안대책 회의록 (학습에 없던 종류)** | 0.63 → 0.97 | 0.42 → 0.69 | 0.67 → 0.91 |
| 보안점검 결과보고서 | 0.71 → 0.96 | 0.28 → 0.71 | 0.58 → 0.88 |
| 보안사고 경위서 | 0.66 → 0.93 | 0.39 → 0.54 | 0.61 → 0.81 |
| 출입기록 점검 결과 | 0.68 → 0.93 | 0.36 → 0.70 | – |
| 비밀문서 관리대장 점검표 | 0.71 → 0.91 | 0.08 → 0.67 | 0.50 → 0.75 |

- 학습에 없던 보안대책 회의록도 다른 문서 종류와 비슷한 점수이다. 다만 회의록도 안건·결정 사항이 번호로 정리된 서식이고 엔티티 사전이 같으므로, 처음 보는 형식에 대한 일반화의 근거로는 약하다.
- 서술형 문장이 많은 보안사고 경위서의 관계(0.54)가 가장 낮다. 한 문장에 인원·매체·문서·장소가 함께 나와 관계 후보가 많기 때문이다.
- 보안 영역별로는 엔티티 0.89~1.00, 관계 0.62~0.72로 큰 차이가 없다.
- 문서 종류 이름과 실제 서식이 다른 경우가 있다. `저장매체 반출입 대장` 17개 중 표 형식의 행이 있는 문서는 7개이고, 나머지는 "대장 확인 결과"를 서술한 보고서이다. 사건과 관계없는 시나리오에도 대장이 배정되어, 작성자가 "기록 없음"을 확인하는 문서로 썼기 때문이다.

## 보조 측정: 사건 서술 구절 추출

이름 있는 엔티티 대신 "보안USB에 비밀 문서를 저장한 채 반출함" 같은 **사건 서술 구절**을 범위로 추출하면 어떤지 평가 경위서 6건에서 측정했다(학습에는 쓰지 않음).

In [ ]:
for name in ("zero_shot", "adapter"):
    print(name, result["conditions"][name]["incident_description"])

| 조건 | 정확 일치 F1 | 정답 6개 중 겹치는 예측이 있는 비율 | 예측한 구절 수 |
|---|---|---|---|
| zero-shot | 0.00 | 33% | 34 |
| 어댑터 | 0.00 | 67% | 28 |

경위서의 모든 문단에서 예측했다. 정확히 일치한 구절은 없고, 문서 6건에서 28~34개의 구절을 내서 대부분 정답이 아니다. 이 레이블은 학습하지 않았고 정답도 6개뿐이라, 이 규모에서는 서술 구절 추출의 가능성을 판단할 근거가 없다. 설계에서 사건을 구절 대신 문서 단위 노드와 레코드 필드로 표현한 방식(방식 B)은 이 결과와 별개로, 레코드 필드 점수(0.842)로 뒷받침된다.

## 문서에서 그래프로

평가 경위서 하나의 정답 그래프와 어댑터가 추출한 그래프를 비교한다.

In [ ]:
example = result["examples"][0]
print(example["text"], "\n")
gold, pred = example["gold"], example["predicted"]
as_set = lambda items: {tuple(item) for item in items}
for kind in ("entities", "relations"):
    g, p = as_set(gold[kind]), as_set(pred[kind])
    print(f"{kind}: correct {len(g & p)}  missed {sorted(g - p)}  extra {sorted(p - g)}")
print("\nrecord gold:", gold["records"][0])
print("record pred:", pred["records"][0])

```text
                         [Document: 경위서 eval-024-1]
                                     │
                         [SecurityIncident: 저장매체 무단 반출, 2026. 4. 22.]
            관련자 │             매체 │          대상 문서 │      장소 │       위반 규정 │
   Person: 일병 윤하율   StorageMedium: 업무용 USB-124   ClassifiedDocument: Ⅲ급 비밀 암호운용 지침
                                                       Facility: 보안자재실   Regulation: 저장매체 통제 지침
   일병 윤하율 ─BELONGS_TO→ 제608공병단 정보과
   일병 윤하율 ─CARRIED_OUT→ 업무용 USB-124 ←STORED_ON─ Ⅲ급 비밀 암호운용 지침   (STORED_ON은 놓침)
   안 소령 ─RESPONSIBLE_FOR→ 업무용 USB-124
   소령 안지후 ─HOLDS_POSITION→ 정보보호 담당관
```

- **사건 레코드는 7개 필드가 모두 정답과 같다.** 사건 유형, 관련자, 매체, 대상 문서, 장소, 위반 규정, 일자가 그대로 채워져 위 그래프의 사건 노드가 완성된다.
- **정답 관계 5개 중 4개를 맞혔다.** `Ⅲ급 비밀 암호운용 지침`을 규정으로 잘못 분류하는 바람에 `-STORED_ON→ 업무용 USB-124`를 놓쳤다(엔티티 타입 오류가 관계로 이어짐).
- 추가로 예측한 관계 4개 중 `안 소령 -CARRIED_OUT→ 업무용 USB-124`(관리 책임자와 반출자 혼동)와 `안 소령 -HOLDS_POSITION→ 관리책임`(일반 명사를 직책으로)은 오답이다. `소령 안지후 -BELONGS_TO→ 제6008공병단 정보과`와 `정보보호 담당관 -POSITION_OF→ 제6008공병단 정보과`는 머리글 내용상 타당하지만 정답지에 없다.
- `안 소령`과 `소령 안지후`, `일병 윤하율`과 `윤 일병`은 같은 사람이지만 표면형이 달라 별도 엔티티로 나온다. 계급·이름 정규화로 하나의 노드로 합치는 단계가 필요하다.

## 결론과 한계

**결론**

| 평가 대상 | 목표 | 결과 | 판단 |
|---|---|---|---|
| 엔티티 | 0.90 | 0.945 (처음 보는 표면형 0.905) | 달성했지만 사전 중복으로 부풀려짐 |
| 레코드 필드 | 0.80 | 0.842 | 달성. 사건 노드의 역할 필드를 구조화 추출로 채울 수 있음 |
| 관계 | 0.80 | 0.656 | 미달. 같은 쌍에 여러 관계를 붙이는 오류가 주된 원인 |
| 사건 유형 | 0.90 | 0.667 (6건) | 판단 불가 |

1. 파이프라인 전체(온톨로지 → 시나리오 → 문서 → 학습 → 채점)가 동작하며, 소규모 가상 데이터로도 학습 효과는 분명하다. 처음 보는 표면형에서도 엔티티 F1이 0.566 → 0.905로 오른다.
2. **관계가 병목이다.** 열람·반출·관리 책임처럼 같은 사람과 대상 사이에 붙을 수 있는 관계를 구분하지 못한다. 학습 관계의 19%가 정렬 문제로 빠진 것도 학습량을 줄였다.
3. 시험 데이터의 구성이 결과를 유리하게 만든다. 엔티티 사전이 작아 평가 데이터의 이름 대부분이 학습에도 나오고, 문서가 모두 비슷한 서식의 감사 보고 문서이다.

**한계**

- 시험 규모(시나리오 30개, 평가 문서 32개)이다. 관계 종류별 평가 정답 2~28개, 선택형 필드 6~17개로 세부 점수의 편차가 크다.
- 학습·평가 데이터를 같은 모델(Claude)이 같은 엔티티 사전으로 작성했다. 평가의 사람 외 엔티티 표면형 91%가 학습에도 나온다.
- 문서 종류가 정형화된 감사 보고 문서에 치우쳐 있고, 관리대장도 실제 표 형식이 아닌 경우가 많다.
- 선택형 필드는 정답 단어가 본문에 적혀 있는지에 좌우되어, 분류 능력을 제대로 재지 못한다.
- 관계 정답지에 일부 누락이 있다(머리글의 점검 대상·소속 등).
- Organization 타입은 데이터에 없어 측정하지 못했다.

**전체 규모로 넓힐 때 고칠 점**

1. **학습과 평가의 엔티티 사전을 나눈다.** 부대·시설·문서·규정·정보체계 이름을 평가 전용으로 따로 두어, 처음 보는 이름에 대한 성능을 잰다.
2. **문서 구성을 실제 감사 자료에 맞춘다.** 결과보고서·경위서는 요약 문서로 두고, 대부분을 표 형식의 관리대장(비밀 관리대장, 열람 기록부, 반출입 대장, 출입 기록부, 계정 관리대장, 인가·교육 현황표)으로 만든다. 사건 유형에 맞는 대장만 배정하고, 정상 기록 사이에 문제 기록을 섞는다.
3. **같은 쌍의 관계를 구분하는 예시를 늘린다.** 열람만 하고 반출하지 않은 기록, 관리 책임자가 직접 접근하지 않은 기록처럼 관계가 갈리는 사례를 의도적으로 넣는다.
4. **관계 정답 작성 지시를 보강한다.** 머리글의 점검 대상·작성자 소속처럼 문서에서 확인할 수 있는 관계는 모두 표시하게 한다.
5. **선택형 필드 정답 단어를 본문에서 뺀 레코드를 만든다.** 분류 능력을 따로 잴 수 있게 한다.
6. 관계 값의 정렬 실패를 줄이도록 이름을 점검하고, Organization이 나오는 시나리오를 추가하고, 같은 사람의 다른 표기를 하나로 합치는 규칙을 둔다.